# 🏋️ EdTech Lead Scoring: Standalone Training & Tuning Pipeline
**Input Dataset:** `edtech_midterm_clean.csv`
**Artifacts Generated:** `model.bin`

In [2]:
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
import xgboost as xgb

## 1. Load Dataset & Feature Sanitation
Αφαιρούμε τα `student_id` (identifier) και `user_country_code` (noise feature) ώστε να μείνουν αυστηρά τα επιλεγμένα χαρακτηριστικά.

In [3]:
df = pd.read_csv('edtech_midterm_clean.csv')

# Sanitation: Drop identifiers & noise
drop_cols = [c for c in ['student_id', 'user_country_code'] if c in df.columns]
if drop_cols:
    df = df.drop(columns=drop_cols)

print(f"Dataset shape for modeling: {df.shape}")
print(f"Target distribution:\n{df['is_converted'].value_counts(normalize=True)}")

Dataset shape for modeling: (2000, 10)
Target distribution:
is_converted
0    0.7325
1    0.2675
Name: proportion, dtype: float64


## 2. Stratified Data Split (60% Train / 20% Val / 20% Test)

In [4]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['is_converted'])
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42, stratify=df_full_train['is_converted'])

y_train = df_train['is_converted'].values
y_val = df_val['is_converted'].values
y_test = df_test['is_converted'].values

del df_train['is_converted']
del df_val['is_converted']
del df_test['is_converted']

# Vectorize with DictVectorizer
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
X_test = dv.transform(df_test.to_dict(orient='records'))

print(f"Encoded features count: {X_train.shape[1]}")

Encoded features count: 17


## 3. Baseline & Hyperparameter Tuning

In [5]:
# Model 1: Logistic Regression Baseline
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
p_lr = lr.predict_proba(X_val)[:, 1]
print(f"Logistic Regression Val AUC: {roc_auc_score(y_val, p_lr):.4f}")

# Model 2: Decision Tree Depth Tuning
for depth in [3, 4, 6, 8]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    p_dt = dt.predict_proba(X_val)[:, 1]
    print(f"Decision Tree (depth={depth}) Val AUC: {roc_auc_score(y_val, p_dt):.4f}")

# Model 3: Random Forest Tuning
for n in [50, 100, 150]:
    rf = RandomForestClassifier(n_estimators=n, max_depth=4, random_state=42)
    rf.fit(X_train, y_train)
    p_rf = rf.predict_proba(X_val)[:, 1]
    print(f"Random Forest (n_est={n}, depth=4) Val AUC: {roc_auc_score(y_val, p_rf):.4f}")

# Model 4: XGBoost Learning Rate Tuning
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

for eta in [0.01, 0.1, 0.3]:
    xgb_params = {'eta': eta, 'max_depth': 4, 'objective': 'binary:logistic', 'eval_metric': 'auc', 'seed': 42}
    model_xgb = xgb.train(xgb_params, dtrain, num_boost_round=100)
    p_xgb = model_xgb.predict(dval)
    print(f"XGBoost (eta={eta}) Val AUC: {roc_auc_score(y_val, p_xgb):.4f}")

c:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Logistic Regression Val AUC: 0.8535
Decision Tree (depth=3) Val AUC: 0.8587
Decision Tree (depth=4) Val AUC: 0.8307
Decision Tree (depth=6) Val AUC: 0.8207
Decision Tree (depth=8) Val AUC: 0.7715
Random Forest (n_est=50, depth=4) Val AUC: 0.8569
Random Forest (n_est=100, depth=4) Val AUC: 0.8579
Random Forest (n_est=150, depth=4) Val AUC: 0.8565
XGBoost (eta=0.01) Val AUC: 0.8537
XGBoost (eta=0.1) Val AUC: 0.8469
XGBoost (eta=0.3) Val AUC: 0.8441


## 4. Retraining on Full Train & Exporting `model.bin`

In [6]:
feature_cols = [c for c in df_full_train.columns if c != 'is_converted']
X_full = dv.fit_transform(df_full_train[feature_cols].to_dict(orient='records'))
X_test_final = dv.transform(df_test.to_dict(orient='records'))
final_features = list(dv.get_feature_names_out())

dfull = xgb.DMatrix(X_full, label=df_full_train['is_converted'].values, feature_names=final_features)
dtest = xgb.DMatrix(X_test_final, label=y_test, feature_names=final_features)

best_xgb_params = {'eta': 0.1, 'max_depth': 4, 'objective': 'binary:logistic', 'eval_metric': 'auc', 'seed': 42}
final_model = xgb.train(best_xgb_params, dfull, num_boost_round=100)

y_pred_test = final_model.predict(dtest)
print(f"FINAL TEST AUC: {roc_auc_score(y_test, y_pred_test):.4f}")

with open('model.bin', 'wb') as f_out:
    pickle.dump((dv, final_model), f_out)
print("Exported 'model.bin' successfully!")

FINAL TEST AUC: 0.8379
Exported 'model.bin' successfully!
